# API Binance

In [ ]:
# !pip install redis

In [ ]:
import ccxt # pip install ccxt
import redis # pip install redis
import pandas as pd # pip install pandas
import json # pip install json
from datetime import datetime # Thu vien built-in

# Thiết lập thông tin tài khoản và sàn giao dịch
exchange = ccxt.binance({ }) # Ko can dua API Key, API Secret

# Thiết lập cặp tiền tệ và thời gian
symbol = 'BTC/USDT'
timeframe = '1m'

# Lấy dữ liệu giá
ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe) # Ham dua Symbol va Timeframe

print(type(ohlcv))
ohlcv

<class 'list'>


[[1753163880000, 117265.49, 117311.38, 117265.48, 117306.25, 10.56694],
 [1753163940000, 117306.26, 117329.28, 117290.78, 117329.27, 15.19591],
 [1753164000000, 117329.28, 117359.16, 117329.28, 117329.37, 25.0217],
 [1753164060000, 117329.36, 117335.61, 117275.47, 117301.78, 22.39468],
 [1753164120000, 117301.78, 117340.9, 117288.47, 117340.9, 10.42516],
 [1753164180000, 117340.9, 117476.5, 117340.9, 117450.28, 40.94459],
 [1753164240000, 117450.29, 117483.2, 117447.46, 117474.67, 19.01298],
 [1753164300000, 117474.66, 117487.18, 117428.96, 117428.96, 4.03416],
 [1753164360000, 117428.96, 117460.0, 117421.63, 117460.0, 11.71798],
 [1753164420000, 117460.0, 117500.0, 117460.0, 117499.99, 2.79348],
 [1753164480000, 117499.99, 117573.83, 117499.99, 117541.05, 9.57932],
 [1753164540000, 117541.06, 117553.45, 117526.6, 117534.37, 13.19762],
 [1753164600000, 117534.37, 117534.37, 117488.19, 117516.92, 4.24767],
 [1753164660000, 117516.92, 117555.0, 117516.91, 117542.2, 2.56275],
 [1753164720

# Chuyen du lieu list sang dataframe

In [ ]:
# Chuyển dữ liệu sang DataFrame
df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
print(df.info())
df

# Chuyen du lieu

In [ ]:
# Chuyển đổi kiểu dữ liệu của giá trị trong DataFrame
df['timestamp'] = df['timestamp'].astype(int)
df['open'] = df['open'].astype(float)
df['high'] = df['high'].astype(float)
df['low'] = df['low'].astype(float)
df['close'] = df['close'].astype(float)
df['volume'] = df['volume'].astype(float)

print(df.info())
df

# Filter

In [ ]:
# Tạo tín hiệu mua dựa trên điều kiện volume lớn hơn 10
df['Buy_Signal'] = df['volume'] > 10
df['Sell_Signal'] = df['volume'] < 10

In [ ]:
df

# Lay record moi nhat de day sang redis

In [ ]:
# Tạo dữ liệu 1 record moi nhat để đẩy qua Redis
data = {
    'symbol': symbol,
    'close': float(df['close'].iloc[-1]),
    'timestamp': int(df['timestamp'].iloc[-1]),
    'InsertDate': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

print(data)

# Day sang bo nho tam Redis

In [ ]:
data

In [ ]:
import redis

# Thiết lập kết nối Redis
redis_client = redis.Redis(host='localhost', port=6379, db=12) # FX, Crypto, Chung khoan

# Tên Hash trong Redis
hash_name = 'Chien luoc Crypto SUI/USDT New 30.05' # OF se doc hash nay

# Ghi từng cặp key-value vào Hash
for key, value in data.items():
    redis_client.hset(hash_name, key, json.dumps(value))

# Doc Redis (Cai nay nam trong OF)

In [ ]:
# Đọc lại dữ liệu từ Redis Hash
retrieved_data = {key.decode(): json.loads(value.decode()) for key, value in redis_client.hgetall(hash_name).items()}
print("Dữ liệu trong Redis Hash:")
print(retrieved_data)

# Lay dict

In [ ]:
hash_name = "OG_ML_CK_MA10, MA20"

In [ ]:
# Đọc lại dữ liệu từ Redis Hash
retrieved_data = {key.decode(): value.decode() for key, value in redis_client.hgetall(hash_name).items()}
print("Dữ liệu trong Redis Hash:")
print(retrieved_data)

# Chien luoc dua vao SMA: Close > SMA10 Va SMA10 > SMA50 => Buy, Close < SMA10 va SMA10 < SMA50 => Sell

In [ ]:
df['SMA2'] = df['close'].rolling(window=2).mean()
df['SMA20'] = df['close'].rolling(window=20).mean()
df['SMA50'] = df['close'].rolling(window=50).mean()

df["Buy_Signal"] = (df['close'] > df['SMA2']) & (df['SMA2'] > df['SMA20'])
df["Sell_Signal"] = (df['close'] < df['SMA2']) & (df['SMA2'] < df['SMA20'])

df

In [ ]:
data = df.iloc[-1].to_dict()
data